# Omar — Data Preparation for Transformer & LoRA
**BESSTIE: Sentiment & Sarcasm Classification across English Varieties**
University of Surrey · Semester 2, 2026

**Prerequisites:** Run Yusrah's `NLP_EDA.ipynb` first to generate `clean_text` and TF-IDF artefacts.

This notebook covers:
1. Vocabulary analysis (Section 1.2) — Jaccard similarity, TF-IDF cosine similarity, heatmaps
2. Linguistic feature analysis — punctuation, CAPS, emoji, sentence length
3. Variety-specific term extraction — Australian, Indian, British
4. Error analysis prep — tricky cases per variety
5. RoBERTa tokenization
6. LoRA subset creation + tokenization (Gemma / Phi-2)

## 0. Setup

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.besstie_data_loader import get_BESSTIE_splits
from src.vocabulary_overlap import VocabularyAnalysis
from src.linguistic_feature_analysis import LinguisticFeatureAnalysis
from src.transformer_tokenization_utils import tokenize_roberta, tokenize_lora_subset

import pandas as pd
import warnings

warnings.filterwarnings("ignore")

print("✅ Imports OK")

In [ ]:
# Load dataset (uses Yusrah's data_loader)
df_all, df_train, df_val, df_test = get_BESSTIE_splits()

# Ensure Sarcasm and Sentiment are int
for df in [df_all, df_train, df_val, df_test]:
    df['Sarcasm'] = df['Sarcasm'].astype(int)
    df['Sentiment'] = df['Sentiment'].astype(float)

print(f'Total rows: {len(df_all):,}')
print(f'Columns: {list(df_all.columns)}')

## 1. Vocabulary Analysis (Section 1.2 Requirement)

In [ ]:
vocab_analyser = VocabularyAnalysis(
    df_all=df_all,
    text_col='text',          # use raw text for vocabulary analysis
    variety_col='variety',
    save_path='./reports'
)

df_jaccard, df_cosine, vocab_per_variety = vocab_analyser.run(save=True)

In [ ]:
# Quick summary of pairwise results
pairs = [('en-AU', 'en-IN'), ('en-AU', 'en-UK'), ('en-IN', 'en-UK')]
print('Pairwise similarity summary:')
print(f'  {"Pair":<20} {"Jaccard":>10} {"TF-IDF Cosine":>15}')
print('  ' + '-'*47)
for a, b in pairs:
    j = df_jaccard.loc[a, b]
    c = df_cosine.loc[a, b]
    print(f'  {a} ↔ {b:<10} {j:>10.4f} {c:>15.4f}')

## 2. Linguistic Feature Analysis

In [ ]:
ling_analyser = LinguisticFeatureAnalysis(
    df_all=df_all,
    text_col='text',
    sarcasm_col='Sarcasm',
    variety_col='variety',
    save_path='./reports'
)

# 2a. Sarcastic vs non-sarcastic comparison (punctuation, CAPS, emoji, sentence length)
df_ling_compare = ling_analyser.compare_sarcasm_features(save=True)
print(df_ling_compare.to_string(index=False))

## 3. Variety-Specific Term Extraction

In [ ]:
# Extract top-30 distinctive terms per variety
# (uses distinctiveness ratio: TF-in-variety / TF-overall)
variety_terms = ling_analyser.extract_variety_terms(top_n=30, save=True)

# Quick look at each variety's top 10
for variety, df_terms in variety_terms.items():
    print(f'\nTop 10 distinctive terms for {variety}:')
    print(df_terms[['term', 'count_in_variety', 'distinctiveness_ratio']].head(10).to_string(index=False))

## 4. Error Analysis Prep

In [ ]:
# Identify 5-10 tricky cases per variety
# Tricky = sarcastic+positive, OR high-punctuation+non-sarcastic
df_tricky = ling_analyser.identify_tricky_cases(n_per_variety=10, save=True)

print(f'\nTotal tricky cases identified: {len(df_tricky)}')
print(df_tricky.groupby(['variety', 'reason']).size())

## 5. Feature Extraction — RoBERTa Tokenization

> **Note:** Loads `clean_text` from Yusrah's pipeline. If `clean_text` column is not present, falls back to `text`.

In [ ]:
# Use clean_text if available (Yusrah's output), else fall back to text
TEXT_COL = 'clean_text' if 'clean_text' in df_train.columns else 'text'
print(f'Using text column: "{TEXT_COL}"')

roberta_datasets = tokenize_roberta(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    text_col=TEXT_COL,
    label_cols=['Sentiment', 'Sarcasm'],
    max_length=128,
    save_path='./tokenized/roberta'
)

print('\nRoBERTa dataset features:', roberta_datasets['train'].features)

## 6. Feature Extraction — LoRA Subset Tokenization

Create **30% stratified subsets per variety** then tokenize with Gemma or Phi-2.

**Change `LORA_MODEL` below to switch between Gemma and Phi-2.**

In [ ]:
# Choose your LoRA model: 'google/gemma-2b'  OR  'microsoft/phi-2'
LORA_MODEL = 'google/gemma-2b'

lora_datasets = tokenize_lora_subset(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    text_col=TEXT_COL,
    label_cols=['Sentiment', 'Sarcasm'],
    model_name=LORA_MODEL,
    max_length=256,
    sample_frac=0.30,
    save_path='./tokenized/lora'
)

print('\nLoRA dataset features:', lora_datasets['train'].features)

In [ ]:
# Optional: run Phi-2 as well
# lora_phi2 = tokenize_lora_subset(
#     df_train=df_train, df_val=df_val, df_test=df_test,
#     text_col=TEXT_COL, label_cols=['Sentiment', 'Sarcasm'],
#     model_name='microsoft/phi-2',
#     max_length=256, sample_frac=0.30,
#     save_path='./tokenized/lora'
# )

## Summary of outputs

| Artefact | Location |
|---|---|
| Vocabulary similarity heatmap | `reports/figures/q1_2_vocabulary_similarity_heatmap.png` |
| Linguistic features chart | `reports/figures/linguistic_features_sarcasm.png` |
| Variety-specific term CSVs | `reports/variety_terms_en_AU.csv` etc. |
| Tricky cases CSV | `reports/tricky_cases_error_analysis.csv` |
| RoBERTa tokenized dataset | `tokenized/roberta/` |
| LoRA tokenized subsets | `tokenized/lora/google_gemma-2b/` |